# Análise de Acromegalia - SIASUS: equidade socioeconômica

Reconstrói a coorte de 2024 (AM + PA, 4.231 pacientes) e gera a tabela de equidade por indicadores municipais (Tabela 3 do manuscrito). As demais análises (demografia, geografia, medicamentos, mapas e figuras) estão em `acromegalia_uniao_medicamentos.ipynb` e `acromegalia_analise_medicamentos.ipynb`.


In [14]:
# Função para gerar nome de arquivo com data atual
# Terceira revisão: todos os artefatos corrigidos são salvos em resultados/third_review/,
# com nomes de arquivos, abas, colunas e valores em inglês (func/translation_en.R)
source("func/translation_en.R")
pasta_resultados <- "../resultados/third_review/"
if (!dir.exists(pasta_resultados)) {
  dir.create(pasta_resultados, recursive = TRUE)
}

gerar_nome_arquivo <- function(indicador) {
  data_atual <- format(Sys.Date(), "%Y%m%d")
  return(paste0(pasta_resultados, data_atual, "_data2024_", nome_arquivo_en(indicador), ".xlsx"))
}

# Função para gerar nome de arquivo PNG
gerar_nome_png <- function(indicador) {
  data_atual <- format(Sys.Date(), "%Y%m%d")
  return(paste0(pasta_resultados, data_atual, "_data2024_", nome_arquivo_en(indicador), ".png"))
}

cat("Funções de formatação de data criadas!\n")
cat("Data atual:", format(Sys.Date(), "%Y%m%d"), "\n")
cat("Pasta de resultados:", pasta_resultados, "\n")


Funções de formatação de data criadas!
Data atual: 20261110 
Pasta de resultados: ../resultados/third_review/ 


## 1. Importação de Dados


In [15]:
# Carregar pacotes necessários
library(tidyverse)
library(lubridate)
library(readr)
library(stringr)
library(purrr)

In [16]:
# Carregar funções personalizadas
source("func/functions.R")


In [17]:
# Definir caminhos dos dados
caminho_dados <- "../data/raw"
caminho_saida <- "../data/processed"

# Criar diretório de saída se não existir
if (!dir.exists(caminho_saida)) {
  dir.create(caminho_saida, recursive = TRUE)
}

### Carregamento dos Dados


In [18]:
# Carregar dados de medicamentos (AM)
data_am <- list.files("../data/processed/medicamentos/", pattern = "\\.csv$", full.names = TRUE) %>%
  map_dfr(~ read_csv(.x, col_types = cols(.default = col_character())))

cat("Dados AM carregados:", nrow(data_am), "registros\n")
cat("Autorizações únicas AM:", n_distinct(data_am$AP_AUTORIZ), "\n")
cat("Autorizações AM com AP_AUTORIZ nulo:", sum(is.na(data_am$AP_AUTORIZ)), "\n")
cat("Pacientes únicos AM:", n_distinct(data_am$AP_CNSPCN), "\n")

Dados AM carregados: 84042 registros
Autorizações únicas AM: 39254 
Autorizações AM com AP_AUTORIZ nulo: 0 
Pacientes únicos AM: 4989 


In [19]:
# Carregar dados de produção ambulatorial (PA)
data_pa <- list.files("../data/processed/producao_ambulatorial/", pattern = "\\.csv$", full.names = TRUE) %>%
  map_dfr(~ read_csv(.x, col_types = cols(.default = col_character())))

cat("Dados PA carregados:", nrow(data_pa), "registros\n")
cat("Autorizações únicas PA:", n_distinct(data_pa$PA_AUTORIZ), "\n")
cat("Autorizações PA com PA_AUTORIZ nulo:", sum(is.na(data_pa$PA_AUTORIZ)), "\n")

Dados PA carregados: 97280 registros
Autorizações únicas PA: 30896 
Autorizações PA com PA_AUTORIZ nulo: 0 


## 2. Preparação dos Dados


In [20]:
# Transformar dados para padronização

## Dados de id da autorização (apenas se não for nulo)
data_am$AP_AUTORIZ <- ifelse(is.na(data_am$AP_AUTORIZ), NA, str_pad(as.character(data_am$AP_AUTORIZ), width = 15, pad = "0"))
data_pa$PA_AUTORIZ <- ifelse(is.na(data_pa$PA_AUTORIZ), NA, str_pad(as.character(data_pa$PA_AUTORIZ), width = 15, pad = "0"))

## Dados da unidade de saúde
data_am$AP_CODUNI <- str_pad(as.character(data_am$AP_CODUNI), width = 7, pad = "0")
data_pa$PA_CODUNI <- str_pad(as.character(data_pa$PA_CODUNI), width = 7, pad = "0")

## Dados do município
data_am$AP_MUNPCN <- str_pad(as.character(data_am$AP_MUNPCN), width = 7, pad = "0")
data_pa$PA_UFMUN <- str_pad(as.character(data_pa$PA_UFMUN), width = 7, pad = "0")

## Dados de data de competência (formato YYYY-MM-DD)
data_am$AP_CMP <- as.Date(paste0(data_am$AP_CMP, "01"), format = "%Y%m%d")
data_pa$PA_CMP <- as.Date(paste0(data_pa$PA_CMP, "01"), format = "%Y%m%d")

## Dados de data de movimentação (formato YYYY-MM-DD)
data_am$AP_MVM <- as.Date(paste0(data_am$AP_MVM, "01"), format = "%Y%m%d")
data_pa$PA_MVM <- as.Date(paste0(data_pa$PA_MVM, "01"), format = "%Y%m%d")

cat("Dados transformados com sucesso!\n")
cat("Formato de datas: YYYY-MM-DD\n")

Dados transformados com sucesso!
Formato de datas: YYYY-MM-DD


## 3. União dos Dados (AM + PA)

**Terceira revisão (itens 5a, 5b, 12.3 do RECORD).** A ligação AM + PA é feita em **nível de autorização** (número da APAC + competência + mês de processamento) e é distinta da **deduplicação em nível de paciente** (CNS criptografado, `AP_CNSPCN`), feita depois na seção 6.

- **AM**: já possui exatamente 1 registro por chave (autorização + competência + processamento) — cada linha é um *registro de dispensação* mensal. Isso é verificado abaixo (`stopifnot`).
- **PA**: pode ter várias linhas por chave (uma por procedimento/linha de produção, inclusive linhas idênticas). Antes da ligação, o PA é **agregado para 1 linha por chave** (campos demográficos: primeiro valor não ausente; quantidades e valores: soma).
- **Cardinalidade esperada**: 1 registro AM : 0 ou 1 chave PA (`relationship = "one-to-one"`). Assim, a ligação **não multiplica** registros de dispensação; o número de linhas da base unificada é igual ao número de registros AM.


In [21]:
# Ligação AM + PA em nível de autorização-competência (terceira revisão, item 5)

# 1. AM: verificar que há exatamente 1 registro por chave (autorização + competência + processamento)
chaves_am <- data_am %>% count(AP_AUTORIZ, AP_CMP, AP_MVM)
stopifnot(all(chaves_am$n == 1))

# 2. PA: agregar para 1 linha por chave ANTES da ligação
data_pa_linhas_duplicadas <- sum(duplicated(data_pa))
data_pa_chave <- data_pa %>%
  mutate(across(c(PA_QTDPRO, PA_QTDAPR, PA_VALPRO, PA_VALAPR), as.numeric)) %>%
  group_by(PA_AUTORIZ, PA_CMP, PA_MVM) %>%
  summarise(
    PA_LINHAS = n(),
    PA_PROC_ID = first(PA_PROC_ID),
    PA_PROC_IDS = paste(sort(unique(PA_PROC_ID)), collapse = ";"),
    PA_IDADE = first(na.omit(PA_IDADE)),
    PA_SEXO = first(na.omit(PA_SEXO)),
    PA_RACACOR = first(na.omit(PA_RACACOR)),
    PA_MUNPCN = first(na.omit(PA_MUNPCN)),
    PA_QTDPRO = sum(PA_QTDPRO, na.rm = TRUE),
    PA_QTDAPR = sum(PA_QTDAPR, na.rm = TRUE),
    PA_VALPRO = sum(PA_VALPRO, na.rm = TRUE),
    PA_VALAPR = sum(PA_VALAPR, na.rm = TRUE),
    .groups = "drop"
  )

# 3. Ligação 1:0..1 (erro se alguma chave casar com mais de uma linha)
dados_unificados <- data_am %>%
  left_join(data_pa_chave, by = c(
    "AP_AUTORIZ" = "PA_AUTORIZ",
    "AP_CMP" = "PA_CMP",
    "AP_MVM" = "PA_MVM"
  ), relationship = "one-to-one") %>%
  mutate(vinculado_pa = !is.na(PA_LINHAS))

stopifnot(nrow(dados_unificados) == nrow(data_am))
n_registros_vinculados <- sum(dados_unificados$vinculado_pa)

# Chaves e autorizações PA sem correspondência em AM (não entram na base)
chaves_pa_sem_am <- data_pa_chave %>%
  anti_join(data_am, by = c("PA_AUTORIZ" = "AP_AUTORIZ", "PA_CMP" = "AP_CMP", "PA_MVM" = "AP_MVM"))
autoriz_pa_sem_am <- setdiff(unique(data_pa$PA_AUTORIZ), unique(data_am$AP_AUTORIZ))

cat("Linhas PA (brutas):", nrow(data_pa), "| linhas exatamente duplicadas:", data_pa_linhas_duplicadas, "\n")
cat("Chaves PA após agregação (autorização-competência-processamento):", nrow(data_pa_chave), "\n")
cat("Registros AM (1 por chave):", nrow(data_am), "\n")
cat("Dados unificados:", nrow(dados_unificados), "registros (igual ao AM: a ligação não multiplica registros)\n")
cat("  • Registros AM com correspondência no PA:", sum(dados_unificados$vinculado_pa), "\n")
cat("  • Registros AM sem correspondência no PA:", sum(!dados_unificados$vinculado_pa), "\n")
cat("  • Chaves PA sem correspondência no AM:", nrow(chaves_pa_sem_am), "\n")
cat("  • Autorizações PA sem nenhuma correspondência no AM:", length(autoriz_pa_sem_am), "\n")
cat("Autorizações únicas unificadas:", n_distinct(dados_unificados$AP_AUTORIZ), "\n")
cat("Pacientes únicos unificados (CNS não ausente):", n_distinct(dados_unificados$AP_CNSPCN, na.rm = TRUE), "\n")


Linhas PA (brutas): 97280 | linhas exatamente duplicadas: 219 
Chaves PA após agregação (autorização-competência-processamento): 75377 
Registros AM (1 por chave): 84042 
Dados unificados: 84042 registros (igual ao AM: a ligação não multiplica registros)
  • Registros AM com correspondência no PA: 74764 
  • Registros AM sem correspondência no PA: 9278 
  • Chaves PA sem correspondência no AM: 613 
  • Autorizações PA sem nenhuma correspondência no AM: 489 
Autorizações únicas unificadas: 39254 
Pacientes únicos unificados (CNS não ausente): 4988 


## 4. Filtro para o Ano de 2024


In [22]:
# Filtrar dados para o ano de 2024 (campo de data: AP_CMP, competência do atendimento)
dados_unificados <- dados_unificados %>%
  filter(AP_CMP >= as.Date("2024-01-01") & AP_CMP <= as.Date("2024-12-31"))

cat("Dados filtrados para 2024:", nrow(dados_unificados), "registros\n")
cat("Autorizações únicas em 2024:", n_distinct(dados_unificados$AP_AUTORIZ), "\n")
cat("✅ Dataset unificado agora contém APENAS dados de 2024\n")


Dados filtrados para 2024: 38993 registros
Autorizações únicas em 2024: 19053 
✅ Dataset unificado agora contém APENAS dados de 2024


### 4.1 Exclusão de registros sem identificador válido do paciente (terceira revisão, item 1)

Registros sem `AP_CNSPCN` não podem ser deduplicados em indivíduos. `n_distinct()` conta o `NA` como uma categoria e `group_by(AP_CNSPCN) %>% slice(1)` juntava todos esses registros em um "paciente" artificial (4.232 = 4.231 + 1). Esses registros são agora **excluídos de todas as análises em nível de paciente**.


In [23]:
# Guardar a base de 2024 antes da exclusão (usada no fluxograma)
dados_2024_todos <- dados_unificados

registros_sem_cns <- sum(is.na(dados_2024_todos$AP_CNSPCN))
autoriz_sem_cns <- n_distinct(dados_2024_todos$AP_AUTORIZ[is.na(dados_2024_todos$AP_CNSPCN)])
autoriz_sem_cns_total <- dados_2024_todos %>%
  group_by(AP_AUTORIZ) %>%
  summarise(todos_sem_cns = all(is.na(AP_CNSPCN)), .groups = "drop") %>%
  filter(todos_sem_cns) %>%
  nrow()

dados_unificados <- dados_2024_todos %>% filter(!is.na(AP_CNSPCN))

cat("=== EXCLUSÃO DE REGISTROS SEM IDENTIFICADOR DO PACIENTE (2024) ===\n")
cat("  • Registros de dispensação excluídos:", registros_sem_cns, "\n")
cat("  • Autorizações com ao menos um registro sem CNS:", autoriz_sem_cns, "\n")
cat("  • Autorizações excluídas por completo (todos os registros sem CNS):", autoriz_sem_cns_total, "\n")
cat("  • Registros remanescentes:", nrow(dados_unificados), "\n")
cat("  • Autorizações remanescentes:", n_distinct(dados_unificados$AP_AUTORIZ), "\n")
cat("  • Pacientes únicos (coorte final):", n_distinct(dados_unificados$AP_CNSPCN), "\n")

tabela_exclusao_cns <- tibble::tibble(
  indicador = c("Registros de dispensação em 2024 (antes da exclusão)",
                "Registros sem CNS do paciente (excluídos)",
                "Autorizações com ao menos um registro sem CNS",
                "Autorizações excluídas por completo",
                "Registros remanescentes",
                "Autorizações remanescentes",
                "Pacientes únicos (coorte final)"),
  valor = c(nrow(dados_2024_todos), registros_sem_cns, autoriz_sem_cns, autoriz_sem_cns_total,
            nrow(dados_unificados), n_distinct(dados_unificados$AP_AUTORIZ),
            n_distinct(dados_unificados$AP_CNSPCN))
)
# Exportação omitida: esta tabela é gerada por acromegalia_analise_medicamentos.ipynb.
print(tabela_exclusao_cns)


=== EXCLUSÃO DE REGISTROS SEM IDENTIFICADOR DO PACIENTE (2024) ===
  • Registros de dispensação excluídos: 434 
  • Autorizações com ao menos um registro sem CNS: 282 
  • Autorizações excluídas por completo (todos os registros sem CNS): 222 
  • Registros remanescentes: 38559 
  • Autorizações remanescentes: 18831 
  • Pacientes únicos (coorte final): 4231 
# A tibble: 7 × 2
  indicador                                            valor
  <chr>                                                <int>
1 Registros de dispensação em 2024 (antes da exclusão) 38993
2 Registros sem CNS do paciente (excluídos)              434
3 Autorizações com ao menos um registro sem CNS          282
4 Autorizações excluídas por completo                    222
5 Registros remanescentes                              38559
6 Autorizações remanescentes                           18831
7 Pacientes únicos (coorte final)                       4231


## 6. Coorte de pacientes únicos (base da análise de equidade)


In [24]:
# Preparar dados para análise demográfica (apenas registros com CNS válido, já filtrados)
dados_2024 <- dados_unificados

# Selecionar APENAS UMA autorização por paciente (AP_CNSPCN) em 2024: a mais recente
# Se houver empate na data, pegar a de maior número de autorização
dados_pacientes_unicos <- dados_2024 %>%
  group_by(AP_CNSPCN) %>%
  arrange(desc(AP_CMP), desc(AP_AUTORIZ)) %>%
  slice(1) %>%
  ungroup()

# Terceira revisão: idade e raça/cor da Tabela 1 vêm do próprio AM (AP_NUIDADE/AP_COIDADE e AP_RACACOR),
# campos completos; antes vinham do PA (PA_IDADE/PA_RACACOR), ausentes quando não havia ligação.
# AP_COIDADE: 4 = idade em anos, 5 = 100 + anos, 0 a 3 = menos de 1 ano
dados_pacientes_unicos <- dados_pacientes_unicos %>%
  mutate(
    IDADE = case_when(AP_COIDADE == "4" ~ as.numeric(AP_NUIDADE),
                      AP_COIDADE == "5" ~ 100 + as.numeric(AP_NUIDADE),
                      AP_COIDADE %in% c("0", "1", "2", "3") ~ 0,
                      TRUE ~ NA_real_),
    RACACOR = AP_RACACOR
  )
cat("Idade ausente (AM):", sum(is.na(dados_pacientes_unicos$IDADE)),
    "| raça/cor ausente (AM):", sum(is.na(dados_pacientes_unicos$RACACOR)), "\n")

n_pacientes <- n_distinct(dados_2024$AP_CNSPCN)

cat("Dados de pacientes únicos de 2024 preparados:\n")
cat("  • Total de registros em 2024:", nrow(dados_2024), "\n")
cat("  • Total de pacientes únicos (AP_CNSPCN válidos):", n_pacientes, "\n")
cat("  • Total de linhas em dados_pacientes_unicos:", nrow(dados_pacientes_unicos), "\n")

# Validação: uma linha por paciente e nenhum CNS ausente
stopifnot(nrow(dados_pacientes_unicos) == n_pacientes, !anyNA(dados_pacientes_unicos$AP_CNSPCN))
cat("✅ dados_pacientes_unicos tem uma linha por paciente com CNS válido\n")


Idade ausente (AM): 0 | raça/cor ausente (AM): 0 
Dados de pacientes únicos de 2024 preparados:
  • Total de registros em 2024: 38559 
  • Total de pacientes únicos (AP_CNSPCN válidos): 4231 
  • Total de linhas em dados_pacientes_unicos: 4231 
✅ dados_pacientes_unicos tem uma linha por paciente com CNS válido


## 7. Análise de equidade socioeconômica

Esta seção constrói a tabela de distribuição dos pacientes por indicadores socioeconômicos municipais e região, utilizando a coorte de pacientes únicos de 2024. Compara o município do **estabelecimento dispensador** (`AP_UFMUN`) com o município de **residência** (`AP_MUNPCN`), utilizando HDI/IDHM e SVI/IVS (2010) e IFDM (2023). As entradas municipais estão em `data/auxiliar`, na raiz do repositório.

A tabela apresenta **contagens de pacientes**, não taxas ponderadas pela população. Não existe uma coluna de região `Unknown`: códigos municipais ausentes ou inválidos interrompem a análise. Pode existir uma *categoria* `Unknown` para indicador municipal sem correspondência ou valor disponível; ela é contabilizada para preservar os totais. A planilha é gravada em `resultados/third_review/`.


### 7.1 Configuração, validação da coorte e códigos municipais

In [25]:
# Célula nova: depende de dados_pacientes_unicos criado na seção 6 deste notebook.
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(readr)
  library(readxl)
  library(openxlsx)
})

if (!exists("dados_pacientes_unicos", inherits = TRUE)) {
  stop("Execute as seções anteriores do notebook até criar 'dados_pacientes_unicos'.")
}

campos_necessarios <- c("AP_CNSPCN", "AP_UFMUN", "AP_MUNPCN")
faltantes <- setdiff(campos_necessarios, names(dados_pacientes_unicos))
if (length(faltantes) > 0) {
  stop("Campos ausentes na coorte: ", paste(faltantes, collapse = ", "))
}

n_equidade <- nrow(dados_pacientes_unicos)
if (n_equidade != 4231L ||
    anyNA(dados_pacientes_unicos$AP_CNSPCN) ||
    dplyr::n_distinct(dados_pacientes_unicos$AP_CNSPCN) != n_equidade) {
  stop("A coorte deve ter 4.231 pacientes, uma linha por AP_CNSPCN válido. ",
       "Confira o processamento e a deduplicação nas seções anteriores.")
}

# O notebook anterior pode preencher AP_MUNPCN com zero à esquerda até 7 caracteres.
# Se for 0 + código de 6 dígitos, remove o zero; se for IBGE de 7 dígitos,
# remove o dígito verificador final. Caso contrário, preserva o código de 6.
ibge6_equidade <- function(x) {
  x <- trimws(as.character(x))
  x <- sub("\\.0+$", "", x)
  x[is.na(x) | x == ""] <- NA_character_
  com_zero <- !is.na(x) & grepl("^0[0-9]{6}$", x)
  x[com_zero] <- substr(x[com_zero], 2, 7)
  sete <- !is.na(x) & grepl("^[0-9]{7}$", x)
  x[sete] <- substr(x[sete], 1, 6)
  x[!grepl("^[0-9]{6}$", x)] <- NA_character_
  x
}

# Harmoniza códigos internos de regiões administrativas do DF com Brasília (IBGE 530010).
# Não modifica códigos de Santa Catarina (420215 = Belmonte; 422000 = Balneário Rincão).
normalizar_df_equidade <- function(codigo) {
  codigo <- as.character(codigo)
  eh_df <- !is.na(codigo) & grepl("^53[0-9]{4}$", codigo)
  codigo[eh_df] <- "530010"
  codigo
}

numero_equidade <- function(x) {
  if (is.numeric(x)) return(as.numeric(x))
  x <- trimws(as.character(x))
  tem_virgula <- !is.na(x) & grepl(",", x, fixed = TRUE)
  x[tem_virgula] <- gsub("\\.", "", x[tem_virgula])
  x[tem_virgula] <- sub(",", ".", x[tem_virgula], fixed = TRUE)
  suppressWarnings(as.numeric(x))
}

regiao_equidade <- function(codigo) {
  uf <- substr(codigo, 1, 2)
  dplyr::case_when(
    uf %in% c("11", "12", "13", "14", "15", "16", "17") ~ "North",
    uf %in% c("21", "22", "23", "24", "25", "26", "27", "28", "29") ~ "Northeast",
    uf %in% c("31", "32", "33", "35") ~ "Southeast",
    uf %in% c("41", "42", "43") ~ "South",
    uf %in% c("50", "51", "52", "53") ~ "Central-West",
    TRUE ~ NA_character_
  )
}

classificar_hdi <- function(x) dplyr::case_when(
  is.na(x) | x < 0 | x > 1 ~ "Unknown",
  x < 0.5 ~ "Very Low", x < 0.6 ~ "Low", x < 0.7 ~ "Medium",
  x < 0.8 ~ "High", TRUE ~ "Very High"
)
classificar_ifdm <- function(x) dplyr::case_when(
  is.na(x) | x < 0 | x > 1 ~ "Unknown",
  x < 0.4 ~ "Critical", x < 0.6 ~ "Low", x < 0.8 ~ "Moderate",
  TRUE ~ "High"
)
classificar_svi <- function(x) dplyr::case_when(
  is.na(x) | x < 0 | x > 1 ~ "Unknown",
  x <= 0.2 ~ "Very Low", x <= 0.3 ~ "Low", x <= 0.4 ~ "Medium",
  x <= 0.5 ~ "High", TRUE ~ "Very High"
)

# Permite executar o notebook a partir de notebooks/ ou da raiz do projeto.
opcoes_auxiliar <- c("../data/auxiliar", "data/auxiliar")
pasta_auxiliar <- opcoes_auxiliar[dir.exists(opcoes_auxiliar)][1]
if (is.na(pasta_auxiliar)) {
  stop("Não encontrei data/auxiliar. Abra o notebook dentro do projeto public-acromegaly.")
}

arquivo_atlas <- file.path(pasta_auxiliar, "atlasivs_dadosbrutos_pt_v2.xlsx")
arquivo_ifdm <- file.path(pasta_auxiliar, "ipeadata[29-08-2025-04-00].csv")
if (!all(file.exists(c(arquivo_atlas, arquivo_ifdm)))) {
  stop("Arquivos auxiliares ausentes: ",
       paste(c(arquivo_atlas, arquivo_ifdm)[!file.exists(c(arquivo_atlas, arquivo_ifdm))],
             collapse = "; "))
}

cat("Coorte validada:", n_equidade, "pacientes. Pasta auxiliar:", pasta_auxiliar, "\n")


Coorte validada: 4231 pacientes. Pasta auxiliar: ../data/auxiliar 


### 7.2 Vínculo aos indicadores municipais e verificação das regiões

In [26]:
# Preserva códigos originais para auditoria e padroniza apenas os códigos do DF.
# Cada paciente permanece uma única vez em cada uma das duas localizações.
municipios_equidade <- dplyr::bind_rows(
  dados_pacientes_unicos %>%
    transmute(Local = "facility", codigo_original = ibge6_equidade(AP_UFMUN)),
  dados_pacientes_unicos %>%
    transmute(Local = "residence", codigo_original = ibge6_equidade(AP_MUNPCN))
) %>%
  mutate(codigo_municipio = normalizar_df_equidade(codigo_original)) %>%
  count(Local, codigo_original, codigo_municipio, name = "n_pacientes") %>%
  mutate(Region = regiao_equidade(codigo_municipio))

# Diagnóstico: quantos pacientes tiveram códigos regionais do DF recodificados.
recodificacoes_df_equidade <- municipios_equidade %>%
  filter(!is.na(codigo_original), codigo_original != codigo_municipio,
         substr(codigo_original, 1, 2) == "53") %>%
  select(Local, codigo_original, codigo_municipio, n_pacientes) %>%
  arrange(Local, codigo_original)
cat("Recodificações dos códigos administrativos do DF para 530010:\n")
print(recodificacoes_df_equidade, n = Inf)
cat("Total de pacientes recodificados por localização:\n")
print(recodificacoes_df_equidade %>%
        group_by(Local) %>% summarise(n = sum(n_pacientes), .groups = "drop"))

invalidos <- municipios_equidade %>%
  filter(is.na(codigo_municipio) | is.na(Region))
if (nrow(invalidos) > 0) {
  stop("Existem municípios ausentes/inválidos na coorte corrigida. ",
       "Nenhum paciente será omitido para eliminar a coluna Unknown. ",
       "Pacientes afetados: ", sum(invalidos$n_pacientes), ".")
}

checagem_locais <- municipios_equidade %>%
  group_by(Local) %>%
  summarise(n = sum(n_pacientes), .groups = "drop")
if (nrow(checagem_locais) != 2L || any(checagem_locais$n != n_equidade)) {
  stop("Erro de contagem: estabelecimento e residência devem somar ", n_equidade, ".")
}

# Atlas IPEA: um registro por município (total da população, ano 2010).
atlas_equidade <- readxl::read_excel(arquivo_atlas) %>%
  filter(municipio != "geral", ano == 2010,
         label_cor == "Total Cor", label_sexo == "Total Sexo",
         label_sit_dom == "Total Situação de Domicílio", udh == "geral") %>%
  transmute(codigo_municipio = normalizar_df_equidade(ibge6_equidade(municipio_6digt)),
            municipio_atlas = as.character(nome_municipio_uf),
            hdi = numero_equidade(idhm), svi = numero_equidade(ivs)) %>%
  filter(!is.na(codigo_municipio))
if (anyDuplicated(atlas_equidade$codigo_municipio)) {
  stop("Atlas: há municípios duplicados após a filtragem.")
}

# IFDM FIRJAN de 2023: padronização do código IBGE antes da vinculação.
ifdm_raw_equidade <- readr::read_csv2(arquivo_ifdm, skip = 1, show_col_types = FALSE)
if (!all(c("Código", "2023") %in% names(ifdm_raw_equidade))) {
  stop("IFDM: não encontrei as colunas 'Código' e '2023'.")
}
ifdm_equidade <- ifdm_raw_equidade %>%
  transmute(codigo_municipio = normalizar_df_equidade(ibge6_equidade(.data[["Código"]])),
            municipio_ifdm = if ("Município" %in% names(ifdm_raw_equidade))
              as.character(.data[["Município"]]) else NA_character_,
            ifdm = numero_equidade(.data[["2023"]])) %>%
  filter(!is.na(codigo_municipio)) %>%
  distinct(codigo_municipio, ifdm, municipio_ifdm)
if (anyDuplicated(ifdm_equidade$codigo_municipio)) {
  stop("IFDM: mais de um valor distinto por município.")
}

base_equidade <- municipios_equidade %>%
  left_join(atlas_equidade, by = "codigo_municipio", relationship = "many-to-one") %>%
  left_join(ifdm_equidade, by = "codigo_municipio", relationship = "many-to-one") %>%
  mutate(HDI = classificar_hdi(hdi),
         IFDM = classificar_ifdm(ifdm),
         SVI = classificar_svi(svi))

if (all(base_equidade$IFDM == "Unknown") ||
    all(base_equidade$HDI == "Unknown") ||
    all(base_equidade$SVI == "Unknown")) {
  stop("Os índices não foram vinculados. Confira os códigos municipais e os arquivos auxiliares.")
}

# Diagnóstico municipal agregado: não divulga qualquer identificador pessoal.
# Mantém Unknown por INDICADOR quando o município existe mas a referência não informa o índice.
diagnostico_equidade <- base_equidade %>%
  select(Local, Region, codigo_original, codigo_municipio,
         municipio_atlas, municipio_ifdm, n_pacientes, HDI, IFDM, SVI) %>%
  pivot_longer(c(HDI, IFDM, SVI), names_to = "Indicator", values_to = "Category") %>%
  filter(Category == "Unknown") %>%
  group_by(Local, Region, codigo_original, codigo_municipio,
           municipio_atlas, municipio_ifdm, Indicator) %>%
  summarise(n_pacientes = sum(n_pacientes), .groups = "drop") %>%
  arrange(Local, Region, codigo_original, Indicator)

cat("Municípios que permaneceram sem classificação, por indicador:\n")
print(diagnostico_equidade, n = Inf)
cat("Conferência dos códigos catarinenses 420215 (Belmonte) e 422000 (Balneário Rincão):\n")
print(base_equidade %>%
        filter(codigo_original %in% c("420215", "422000")) %>%
        select(Local, codigo_original, codigo_municipio,
               municipio_atlas, municipio_ifdm, n_pacientes, HDI, IFDM, SVI),
      n = Inf)

cat("Pacientes por localização:\n")
print(checagem_locais)
cat("Pacientes sem classificação municipal por índice (pode haver Unknown como categoria):\n")
print(base_equidade %>%
        select(Local, n_pacientes, HDI, IFDM, SVI) %>%
        pivot_longer(c(HDI, IFDM, SVI), names_to = "Indicator", values_to = "Category") %>%
        group_by(Local, Indicator) %>%
        summarise(n_unknown = sum(n_pacientes[Category == "Unknown"]), .groups = "drop"))


Recodificações dos códigos administrativos do DF para 530010:
# A tibble: 7 × 4
  Local     codigo_original codigo_municipio n_pacientes
  <chr>     <chr>           <chr>                  <int>
1 residence 530020          530010                     5
2 residence 530040          530010                     9
3 residence 530060          530010                     1
4 residence 530110          530010                     1
5 residence 530140          530010                     1
6 residence 530150          530010                     1
7 residence 530180          530010                     9
Total de pacientes recodificados por localização:
# A tibble: 1 × 2
  Local         n
  <chr>     <int>
1 residence    27


ℹ Using "','" as decimal and "'.'" as grouping mark. Use `read_delim()` for more control.

New names:
• `` -> `...9`


Municípios que permaneceram sem classificação, por indicador:
# A tibble: 3 × 8
  Local   Region codigo_original codigo_municipio municipio_atlas municipio_ifdm
  <chr>   <chr>  <chr>           <chr>            <chr>           <chr>         
1 reside… South  422000          422000           NA              Balneário Rin…
2 reside… South  422000          422000           NA              Balneário Rin…
3 reside… South  430583          430583           Coqueiro Baixo… Coqueiro Baixo
# ℹ 2 more variables: Indicator <chr>, n_pacientes <int>
Conferência dos códigos catarinenses 420215 (Belmonte) e 422000 (Balneário Rincão):
# A tibble: 2 × 9
  Local     codigo_original codigo_municipio municipio_atlas municipio_ifdm  
  <chr>     <chr>           <chr>            <chr>           <chr>           
1 residence 420215          420215           Belmonte (SC)   Belmonte        
2 residence 422000          422000           NA              Balneário Rincão
# ℹ 4 more variables: n_pacientes <int>, HDI

### 7.3 Construção e validação da tabela de equidade

In [27]:
regioes_equidade <- c("Southeast", "Northeast", "South", "Central-West", "North")
ordem_equidade <- tibble::tribble(
  ~Indicator, ~Category, ~ordem,
  "HDI", "Very Low", 1L, "HDI", "Low", 2L, "HDI", "Medium", 3L,
  "HDI", "High", 4L, "HDI", "Very High", 5L, "HDI", "Unknown", 6L,
  "IFDM", "Critical", 1L, "IFDM", "Low", 2L, "IFDM", "Moderate", 3L,
  "IFDM", "High", 4L, "IFDM", "Unknown", 5L,
  "SVI", "Very Low", 1L, "SVI", "Low", 2L, "SVI", "Medium", 3L,
  "SVI", "High", 4L, "SVI", "Very High", 5L, "SVI", "Unknown", 6L
)

observado_equidade <- base_equidade %>%
  select(Local, Region, n_pacientes, HDI, IFDM, SVI) %>%
  pivot_longer(c(HDI, IFDM, SVI), names_to = "Indicator", values_to = "Category") %>%
  group_by(Local, Region, Indicator, Category) %>%
  summarise(n = sum(n_pacientes), .groups = "drop")

# Mantém categorias observadas em pelo menos uma localização; inclui zeros por região.
categorias_equidade <- ordem_equidade %>%
  left_join(observado_equidade %>%
              group_by(Indicator, Category) %>%
              summarise(total = sum(n), .groups = "drop"),
            by = c("Indicator", "Category")) %>%
  filter(!is.na(total) & total > 0) %>%
  select(-total)

detalhes_equidade <- tidyr::crossing(
  Local = c("facility", "residence"),
  Region = regioes_equidade,
  categorias_equidade
) %>%
  left_join(observado_equidade,
            by = c("Local", "Region", "Indicator", "Category")) %>%
  mutate(n = replace_na(n, 0L))

totais_equidade <- municipios_equidade %>%
  group_by(Local, Region) %>%
  summarise(n = sum(n_pacientes), .groups = "drop") %>%
  tidyr::crossing(Indicator = c("HDI", "IFDM", "SVI"))

totais_equidade <- tidyr::crossing(
  Local = c("facility", "residence"),
  Region = regioes_equidade,
  Indicator = c("HDI", "IFDM", "SVI")
) %>%
  left_join(totais_equidade, by = c("Local", "Region", "Indicator")) %>%
  mutate(n = replace_na(n, 0L), Category = "Total", ordem = 0L)

# Conferência: a soma das categorias deve reproduzir o total de cada região.
conferencia_equidade <- bind_rows(
  totais_equidade %>% select(Local, Region, Indicator, Category, n),
  detalhes_equidade %>% select(Local, Region, Indicator, Category, n)
) %>%
  group_by(Local, Region, Indicator) %>%
  summarise(total = n[Category == "Total"][1],
            estratos = sum(n[Category != "Total"]), .groups = "drop")
if (any(conferencia_equidade$total != conferencia_equidade$estratos)) {
  stop("Os estratos de equidade não reconciliam com os totais regionais.")
}

longa_equidade <- bind_rows(
  totais_equidade %>% select(Local, Region, Indicator, Category, ordem, n),
  detalhes_equidade %>% select(Local, Region, Indicator, Category, ordem, n)
) %>%
  mutate(coluna = paste(Local, Region, sep = "_")) %>%
  select(Indicator, Category, ordem, coluna, n)

fac_cols <- paste0("facility_", regioes_equidade)
res_cols <- paste0("residence_", regioes_equidade)
tabela_equidade_final <- longa_equidade %>%
  pivot_wider(names_from = coluna, values_from = n, values_fill = 0L) %>%
  mutate(facility_Total = rowSums(across(all_of(fac_cols))),
         residence_Total = rowSums(across(all_of(res_cols))),
         indicador_ordem = match(Indicator, c("HDI", "IFDM", "SVI"))) %>%
  arrange(indicador_ordem, ordem) %>%
  select(Indicator, Category, all_of(fac_cols), facility_Total,
         all_of(res_cols), residence_Total)

for (ind in c("HDI", "IFDM", "SVI")) {
  fatia <- tabela_equidade_final %>% filter(Indicator == ind)
  linha_total <- fatia %>% filter(Category == "Total")
  for (col in c("facility_Total", "residence_Total")) {
    if (nrow(linha_total) != 1L || linha_total[[col]] != n_equidade ||
        sum(fatia[[col]][fatia$Category != "Total"]) != n_equidade) {
      stop("A tabela de equidade não reconcilia com a coorte de ", n_equidade,
           " pacientes para ", ind, " / ", col)
    }
  }
}

cat("Tabela de equidade validada: ", n_equidade,
    " pacientes no total de estabelecimento e no total de residência.\n", sep = "")
print(tabela_equidade_final, n = Inf)


Tabela de equidade validada: 4231 pacientes no total de estabelecimento e no total de residência.
# A tibble: 19 × 14
   Indicator Category  facility_Southeast facility_Northeast facility_South
   <chr>     <chr>                  <int>              <int>          <int>
 1 HDI       Total                   2320                784            732
 2 HDI       Low                        0                  0              0
 3 HDI       Medium                    13                 50              0
 4 HDI       High                    1002                734            373
 5 HDI       Very High               1305                  0            359
 6 HDI       Unknown                    0                  0              0
 7 IFDM      Total                   2320                784            732
 8 IFDM      Critical                   0                  0              0
 9 IFDM      Low                        7                 32              0
10 IFDM      Moderate                 539     

### 7.4 Exportação para Excel no formato do manuscrito

In [28]:
# Reutiliza a pasta resultados/third_review/ já configurada no início do notebook.
saida_equidade <- if (exists("pasta_resultados", inherits = TRUE)) {
  file.path(pasta_resultados, "tabela_equidade.xlsx")
} else {
  file.path("../resultados/third_review", "tabela_equidade.xlsx")
}
dir.create(dirname(saida_equidade), showWarnings = FALSE, recursive = TRUE)

wb_equidade <- openxlsx::createWorkbook()
openxlsx::addWorksheet(wb_equidade, "Equidade", gridLines = FALSE)
openxlsx::writeData(
  wb_equidade, "Equidade",
  "Treated-patient distribution by municipal socioeconomic indicator and region (SUS, 2024)",
  startRow = 1, startCol = 1
)

openxlsx::writeData(wb_equidade, "Equidade", "Indicator", startRow = 3, startCol = 1)
openxlsx::writeData(wb_equidade, "Equidade", "Category", startRow = 3, startCol = 2)
openxlsx::writeData(wb_equidade, "Equidade", "Health Facility Municipality", startRow = 3, startCol = 3)
openxlsx::writeData(wb_equidade, "Equidade", "Patient's Municipality of Residence", startRow = 3, startCol = 9)
openxlsx::mergeCells(wb_equidade, "Equidade", cols = 1, rows = 3:4)
openxlsx::mergeCells(wb_equidade, "Equidade", cols = 2, rows = 3:4)
openxlsx::mergeCells(wb_equidade, "Equidade", cols = 3:8, rows = 3)
openxlsx::mergeCells(wb_equidade, "Equidade", cols = 9:14, rows = 3)
openxlsx::writeData(
  wb_equidade, "Equidade",
  matrix(rep(c(regioes_equidade, "Total"), 2), nrow = 1),
  startRow = 4, startCol = 3, colNames = FALSE
)

publicacao_equidade <- tabela_equidade_final
publicacao_equidade$Indicator[duplicated(publicacao_equidade$Indicator)] <- ""
openxlsx::writeData(wb_equidade, "Equidade", publicacao_equidade,
                    startRow = 5, startCol = 1, colNames = FALSE)

estilo_cab_equidade <- openxlsx::createStyle(
  textDecoration = "bold", halign = "center", valign = "center",
  border = "Bottom", wrapText = TRUE
)
estilo_n_equidade <- openxlsx::createStyle(numFmt = "#,##0", halign = "center")
estilo_total_equidade <- openxlsx::createStyle(textDecoration = "bold", border = "Top")
openxlsx::addStyle(wb_equidade, "Equidade", estilo_cab_equidade,
                   rows = 3:4, cols = 1:14, gridExpand = TRUE)
openxlsx::addStyle(wb_equidade, "Equidade", estilo_n_equidade,
                   rows = 5:(nrow(tabela_equidade_final) + 4), cols = 3:14,
                   gridExpand = TRUE, stack = TRUE)
openxlsx::addStyle(wb_equidade, "Equidade", estilo_total_equidade,
                   rows = which(tabela_equidade_final$Category == "Total") + 4,
                   cols = 1:14, gridExpand = TRUE, stack = TRUE)
openxlsx::setColWidths(wb_equidade, "Equidade", cols = 1:2, widths = c(13, 17))
openxlsx::setColWidths(wb_equidade, "Equidade", cols = 3:14, widths = 14)
openxlsx::setRowHeights(wb_equidade, "Equidade", rows = 3:4, heights = c(27, 32))

nota_equidade <- paste0(
  "HDI: Human Development Index (2010); IFDM: FIRJAN Municipal Development Index (2023); ",
  "SVI: Social Vulnerability Index (2010). Values are patient counts, not population-adjusted rates. ",
  "Municipalities of facility and residence were derived from AP_UFMUN and AP_MUNPCN, respectively. ",
  "DF administrative-region codes were harmonized to Brasília (530010). ",
  "An 'Unknown' indicator category denotes unavailable municipal classification, not unknown residence. ",
  "See the Diagnostico worksheet for unmatched municipality codes."
)
linha_nota_equidade <- nrow(tabela_equidade_final) + 7
openxlsx::writeData(wb_equidade, "Equidade", nota_equidade,
                    startRow = linha_nota_equidade, startCol = 1)
openxlsx::mergeCells(wb_equidade, "Equidade", cols = 1:14, rows = linha_nota_equidade)
openxlsx::addStyle(wb_equidade, "Equidade",
                   openxlsx::createStyle(wrapText = TRUE, fontSize = 9),
                   rows = linha_nota_equidade, cols = 1, stack = TRUE)
openxlsx::setRowHeights(wb_equidade, "Equidade", rows = linha_nota_equidade, heights = 42)
openxlsx::freezePane(wb_equidade, "Equidade", firstActiveRow = 5, firstActiveCol = 3)

# A segunda aba discrimina municípios não classificados e seus códigos originais.
openxlsx::addWorksheet(wb_equidade, "Diagnostico", gridLines = FALSE)
if (nrow(diagnostico_equidade) > 0) {
  openxlsx::writeData(wb_equidade, "Diagnostico", diagnostico_equidade)
  openxlsx::setColWidths(wb_equidade, "Diagnostico",
                         cols = seq_len(ncol(diagnostico_equidade)), widths = "auto")
} else {
  openxlsx::writeData(wb_equidade, "Diagnostico",
                      "Todos os pacientes receberam classificação em todos os indicadores.")
}
openxlsx::saveWorkbook(wb_equidade, saida_equidade, overwrite = TRUE)
cat("Tabela de equidade salva em:", normalizePath(saida_equidade, winslash = "/"), "\n")


Tabela de equidade salva em: /Users/daniellybx/Documents/Projetos LEFAR:FS:UnB - CNPq/public-acromegaly/resultados/third_review/tabela_equidade.xlsx 
